# Overview

Simple demo with flu model with "toy" (not necessarily realistic or finalized) parameters

We go over
- How to set up subpopulation models and metapopulation models.
- How to run accept-reject algorithm.
- How to run experiments to organize output of multiple replications. 

# Imports

In [1]:
# Misc: suppress SSE4.2 deprecation warnings
# This is not an error, just a warning about future hardware compatibility (not relevant in general)
import os
os.environ["MKL_DEBUG_CPU_TYPE"] = "5" 

from pathlib import Path
import numpy as np
import pandas as pd

# Import city-level transmission base components module
import clt_toolkit as clt

# Import flu model module, which contains customized subclasses
import flu_core as flu

import json

# Reading Input Files

In [2]:
# Note that this can be cumbersome due to the complexity of the model 
# Users may prefer to create the model inputs or even the models themselves in a separate .py file and import them! 

# Obtain path to folder with JSON input files
base_path = clt.utils.PROJECT_ROOT / "flu_instances" / "texas_input_files"

# Get filepaths
compartments_epi_metrics_init_vals_filepath = base_path / "init_vals.json"
subpop_params_filepath = base_path / "common_subpop_params.json"
mixing_params_filepath = base_path / "mixing_params.json"
simulation_settings_filepath = base_path / "simulation_settings.json"

calendar_df = pd.read_csv(base_path / "school_work_calendar.csv", index_col=0)
humidity_df = pd.read_csv(base_path / "absolute_humidity_austin_2023_2024.csv", index_col=0)
vaccines_df = pd.read_csv(base_path / "daily_vaccines_constant.csv", index_col=0)
mobility_modifier_df = pd.read_csv(base_path / "mobility_modifier.csv", index_col=0)

schedules_info = flu.FluSubpopSchedules(absolute_humidity=humidity_df,
                                        flu_contact_matrix=calendar_df,
                                        daily_vaccines=vaccines_df,
                                        mobility_modifier=mobility_modifier_df)

# Read in files as dictionaries and dataframes.
# Note that we can also create these dictionaries directly
#   rather than reading from a predefined input data file.
state = clt.make_dataclass_from_json(compartments_epi_metrics_init_vals_filepath,
                                     flu.FluSubpopState)
params = clt.make_dataclass_from_json(subpop_params_filepath, flu.FluSubpopParams)
mixing_params = clt.make_dataclass_from_json(mixing_params_filepath, flu.FluMixingParams)
settings = clt.make_dataclass_from_json(simulation_settings_filepath, flu.SimulationSettings)

# Create two independent bit generators
bit_generator = np.random.MT19937(88888)
jumped_bit_generator = bit_generator.jumped(1)

# Creating Subpopulation Models

In [3]:
# Create two subpopulation models, one for the north
#   side of the city and one for the south side of the city
# In this case, these two (toy) subpopulations have the
#   same demographics, initial compartment and epi metric values,
#   fixed parameters, and school-work calendar.
# If we wanted the "north" subpopulation and "south"
#   subpopulation to have different aforementioned values,
#   we could read in two separate sets of files -- one
#   for each subpopulation.
north = flu.FluSubpopModel(state,
                           params,
                           settings,
                           np.random.Generator(bit_generator),
                           schedules_info,
                           name="north")

south = flu.FluSubpopModel(state,
                           params,
                           settings,
                           np.random.Generator(jumped_bit_generator),
                           schedules_info,
                           name="south")

# If a metapopulation model has N subpopulations that have most of their parameters in common,
#    users can make a JSON file with common parameters, and N smaller JSON files with
#    parameters that are specific to each subpopulation.
# We recommend using `clt_toolkit / utils` functions `updated_dataclass()` and `updated_dict()` for this.

/Users/susan/Pycharm/CLT_BaseModel/flu_core/flu_components.py:497: UserWarning: Vaccine immunity reset date is set as 08/01. 
Initial vaccine-induced immunity value is being adjusted by resetting immunity to 0 at that date, and by taking into account vaccines administered after this date, and before simulation start date.
  warnings.warn(msg)
/Users/susan/Pycharm/CLT_BaseModel/flu_core/flu_components.py:497: UserWarning: Vaccine immunity reset date is set as 08/01. 
Initial vaccine-induced immunity value is being adjusted by resetting immunity to 0 at that date, and by taking into account vaccines administered after this date, and before simulation start date.
  warnings.warn(msg)


# Modifying Parameters

In [4]:
# We can also manually change a fixed parameter value
#   after we have created a SubpopModel -- like so...
# Note that this is quite a large and unrealistic value of
#   beta_baseline, but we'll use this to create
#   a dramatic difference between the two subpopulations.
south.params = clt.updated_dataclass(south.params, {"beta_baseline": 10})

# The structure of the code allows us to access
#   the current state and fixed parameters of each
#   subpopulation model.
# For example, here we print out the fixed parameter
#   value for beta_baseline for the "south" subpopulation.
print(south.params.beta_baseline)

10


# Create Metapopulation Model

In [5]:
# Combine two subpopulations into one metapopulation model (travel model)
flu_demo_model = flu.FluMetapopModel([north, south],
                                     mixing_params)

# Experiments

In [6]:
# Create an experiment class to record "HR", "HD", and "D" compartment values
experiment = clt.Experiment(model = flu_demo_model, 
                            state_variables_to_record = ["HR", "HD", "D"], 
                            database_filename = "flu_demo_H_D_results.db")

# Run the simulation for 10 independent replications, for 100 days, and save the values of
#    "HR", "HD", and "D" every 7 days. Save the results in a CSV file. 
experiment.run_static_inputs(num_reps = 10, 
                             simulation_end_day = 100, 
                             days_between_save_history = 7, 
                             results_filename = "flu_demo_H_D_results.csv")

In [7]:
# Here's the CSV we just created -- pretty nifty! :) 
H_D_results_df = pd.read_csv("flu_demo_H_D_results.csv", index_col=0)

H_D_results_df.head()

,subpop_name,state_var_name,age_group,risk_group,rep,timepoint,value
0,north,HR,0,0,0,7,10.0
1,north,HR,1,0,0,7,35.0
2,north,HR,2,0,0,7,105.0
3,north,HR,3,0,0,7,68.0
4,north,HR,4,0,0,7,261.0


In [8]:
# `run_static_inputs` also creates a SQL database with the large-scale output from our experiment.
!ls *.db*

flu_demo_H_D_results.db


In [9]:
# Build dataframe of "HR" state based on experiment run above (simulated 100 days, state saved every 7 days).
# See `get_state_var_df()` docstring for details. 
# Because we do not specify `subpop_name`, `age_group`, or `risk_group`, 
#     compartment values are summed across subpopulations, ages, and risk groups.
experiment.get_state_var_df(state_var_name = "HR",
                            results_filename = "flu_demo_aggregated_H_results.csv")

timepoint,7,14,21,28,35,42,49,56,63,70,77,84,91,98,100
rep,,,,,,,,,,,,,,,
0,27427.0,188337.0,180729.0,102691.0,53796.0,31124.0,21247.0,17130.0,14852.0,13383.0,12180.0,11260.0,10662.0,10104.0,9843.0
1,27403.0,188645.0,180943.0,102610.0,53866.0,31224.0,21533.0,17158.0,14860.0,13217.0,12049.0,11131.0,10353.0,9941.0,9795.0
2,27619.0,188407.0,179779.0,102748.0,53596.0,31171.0,21750.0,17144.0,14772.0,13377.0,12297.0,11457.0,10625.0,10087.0,9939.0
3,27338.0,187878.0,180079.0,102909.0,54042.0,30788.0,21475.0,17211.0,15012.0,13364.0,12110.0,11315.0,10467.0,10018.0,9858.0
4,27082.0,187372.0,179954.0,102836.0,53594.0,31063.0,21607.0,17135.0,14907.0,13359.0,12237.0,11248.0,10600.0,10041.0,9836.0
5,27500.0,188244.0,180666.0,102867.0,53789.0,31080.0,21597.0,17079.0,14832.0,13250.0,12071.0,11349.0,10621.0,10064.0,9869.0
6,26503.0,186859.0,180507.0,102774.0,53636.0,31187.0,21645.0,17362.0,15134.0,13241.0,12266.0,11229.0,10521.0,9996.0,9885.0
7,27081.0,188646.0,180995.0,103469.0,54093.0,31122.0,21479.0,17154.0,14800.0,13068.0,12068.0,11033.0,10681.0,10069.0,9893.0
8,26958.0,188178.0,180935.0,102934.0,53984.0,31291.0,21553.0,17255.0,14865.0,13195.0,11994.0,11073.0,10435.0,9968.0,9798.0


In [10]:
# Delete temporary demo files
do_delete_demo_files = True
if do_delete_demo_files:
    temp_files = ["flu_demo_H_D_results.csv",
                  "flu_demo_H_D_results.db",
                  "flu_demo_aggregated_H_results.csv"]
    
    for file in temp_files:
        Path(file).unlink()
        